# TI-City 18<br>
Morris pass-1 design generator (10 groups / 25 variables / 66 runs)

Generates the complete parameter sets for the Morris group-screening design agreed on 2026-11-25 (supersedes the 11-group / 24-var sketch and the 27-var factor-scaling draft; see canvas `ta-city-calib-batch-and-morris`, spec v3):

- **66 runs** = 6 trajectories x (10 groups + 1); Morris steps one GROUP per jump, so the cost is set by the group count, not the variable count.
- Sampled values are baked into one **weights CSV per run** (`weights_s_m001.csv` ... `weights_s_m066.csv`, 6 agent rows x 17 columns, cloned from the s0 template) plus one row per run in **`scenarios_morris.csv`** (schema v3, household-number excluded/`run-num` conventions cloned from s0, individual `run-num` 129-194).
- **Frozen / excluded**: household-number (quantity already matched at Phase 0), maxof, switches, airport (no utility term in v41), landvalue thresholds (hardcoded), structureplan (paired runs later).
- **Common random numbers**: all 66 runs use run-seed 11; seeds 22/33 only for confirmation runs of top-ranked groups.
- Realized class shares are NORMALIZED (not clipped) so every realization is feasible and elementary effects stay valid (see writer rule 3 in the spec).

# Imports

In [ ]:
import sys
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import rasterio
import numpy as np

from SALib.sample import morris as morris_sample
from SALib.analyze import morris as morris_analyze

from ti_city_00_helper_functions import *

print('python', sys.version.split()[0], '| SALib ok')

# init

In [ ]:
case_city = 'JAK' #MUM MAN JAK

# repo-side paths (adjust to run location; notebook expects repo root as cwd)
ti_city_model_path = Path(f'../model/{case_city}/')
ti_city_in_path    = Path(f'../model/{case_city}/data/in/')
docs_dir           = Path('../docs/')

# templates (cloned for every run)
template_weights_path   = ti_city_in_path / 'weights_s0.csv'
template_scenarios_path = ti_city_in_path / 'scenarios.csv'

# template scenario row to clone
template_scenario_name = 's0'

case_city = 'JAK' #MUM MAN JAK
drop_path = r'C:\\Sciebo\\05_GIS\\' # adjust this to match the path on your computer, e.g. 'D:\\Dropbox\\x\\PostDoc\\GIS' or 'C:\\Users\\x\\Dropbox\\x\\PostDoc\\GIS'
git_path = Path(r'C:\\GIT\\ti-city-model') # adjust this to match the path on your computer, e.g. 'D:\\GIT\\ti-city-model' or 'C:\\Users\\x\\GIT\\ti-city-model'
external_data_path = Path(r'C:\\Sciebo\\00_data') # adjust this to match the path on your computer, e.g. 'D:\\Dropbox\\x\\PostDoc\\00_data' or 'C:\\Users\\x\\Dropbox\\x\\PostDoc\\00_data'
## ascii150_arcgis_path = Path(r"D:\\Dropbox\\x\\PostDoc\ASCII_150m") deprecated, ignore

#input a vector and a raster file for each city
AOI_path = Path(f'{drop_path}\\{case_city}\\processed\\')
AOI_file = f'{case_city}_LIM_AOI_reference_150m_A.shp' # APS: updated AOI 01.08.2024

AOI_gdf = gpd.read_file(AOI_path / AOI_file) #.to_crs(epsg=4326)
AOI_gdf_4326 = AOI_gdf.to_crs(epsg=4326) 

ref_raster_path = Path(f'../data/processed/{case_city}_LIM_reference_AOI_150m.tif')
ref_raster_150_path = Path(f'../data/processed/{case_city}_LIM_reference_AOI_150m.tif')

raw_path = Path(f'\\{case_city}\\raw\\')
interim_path = Path(drop_path + f'{case_city}\\interim\\')
processed_path = Path(drop_path + f'{case_city}\\processed\\')
external_path = Path(drop_path + f'{case_city}\\external\\')
model_inputs_30m_path = Path(drop_path + f'{case_city}\\model_inputs\\TIFF_30m\\')
model_inputs_150m_path = Path(drop_path + f'{case_city}\\model_inputs\\TIFF_150m\\')
model_inputs_SLEUTH_150m_path = Path(drop_path + f'{case_city}\\model_inputs\\TIFF_SLEUTH_150m')
model_outputs_path = Path(drop_path + f'{case_city}\\model_outputs\\') 
ascii150_path = Path(drop_path + f'\\{case_city}\\model_inputs\\ASCII_150m')
ascii150_SLEUTH_path = Path(drop_path + f'\\{case_city}\\model_inputs\\ASCII_SLEUTH_150m')
ti_city_model_path = Path(f'../model/{case_city}/')
ti_city_in_path = Path(f'../model/{case_city}/data/in/')
ti_city_out_path = Path(f'../model/{case_city}/data/out/')
ref_raster_sleuth_path = model_inputs_SLEUTH_150m_path / (case_city + '_URB_SLEUTH_input_2022.tif' )

if case_city == 'MUM':
    gadm_var = 'NAME_3'
    calibration_years = [2015,2025]
    simulation_years = [2025, 2050]

if case_city == 'MAN':    
    gadm_var = 'NAME_2'
    calibration_years = [2015,2025]
    simulation_years = [2025, 2050]

if case_city == 'JAK':
    gadm_var = 'NAME_3'
    calibration_years = [2000,2025]
    simulation_years = [2025, 2050]

# read the reference raster
with rasterio.open(ref_raster_path,'r') as src: # APS 20.08.2025 using the 150 m raster
    ref_raster = src
    ref_meta = src.meta
    ref_height, ref_width, ref_area = get_transform(ref_raster)

# suppress deprecation warnings
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn

# first, let's inform the scenario folders
# APS: 15.05.2026 I updated the folder and file naming convention to allow for individual results per tick
# example: "D:\GIT\ti-city-model\model\JAK\data\out\R037\expansion_income_t0.asc"


# let's build dictionaries to support the loading functions
agent_class_dict = {'na':0,'low': 1, 'mid': 2, 'high': 3, 'deprived': 4, 'urban_2022': 5}
#APS 11.08.2026 previously: ['std', 'ssp1', 'ssp2', 'ssp3']
#APS 22.10.2026 previously: ['S1', 'S2', 'S3', 'S4'] 
ascii_files = [] # initialize the list of ASCII files

print(case_city, agent_class_dict)

# 1. Morris problem (10 groups / 25 variables / spec v3)

Group assignments follow the canvas spec v3. `groups` maps each variable to the group that jumps together. The RED agents receive the same 14 sampled weights as every agent class; their only RED-specific sampled knobs are the two `rad` columns and the two scenario scalars (the `w_red_mi`/`w_red_hi` multiplicative-factor draft is REVOKED: scaling the nominal RED weight row could push `exposed` outside the signed instrument, e.g. -3 x 2 = -6).

In [15]:
morris_problem = {
    'num_vars': 25,
    'names': [
        'w_road',
        'w_bu', 'w_density',
        'w_suburban', 'w_cbd', 'w_attractive',
        'w_mall', 'w_markets', 'w_schools', 'w_health',
        'w_random', 'w_water',
        'w_exposed',                                   # SIGNED: disamenity can flip to amenity
        'slope_coefficient', 'critical_slope', 'w_nom_slope',
        'share_dp', 'share_li', 'share_mi', 'w_neighbour',
        'development_control',
        'red_rad_mi', 'red_rad_hi',
        'estate_developer_number', 'pct_high_end_developers',
    ],
    'groups': [
        'road',
        'existing_urban', 'existing_urban',
        'social', 'social', 'social',
        'facilities', 'facilities', 'facilities', 'facilities',
        'disamenity_random', 'disamenity_random',
        'exposure',
        'terrain', 'terrain', 'terrain',
        'class_shares', 'class_shares', 'class_shares', 'class_shares',
        'planning',
        'red', 'red', 'red', 'red',
    ],
    'bounds': [
        [0, 10],
        [0, 10], [0, 10],
        [0, 10], [0, 10], [0, 10],
        [0, 10], [0, 10], [0, 10], [0, 10],
        [0, 10], [0, 10],
        [-5, 5],
        [20, 100], [5, 30], [0, 10],
        [5, 20], [30, 60], [10, 35], [0, 10],
        [30, 90],
        [1, 5], [1, 5],
        [100, 600], [10, 60],
    ],
}

n_groups       = len(set(morris_problem['groups']))
n_trajectories = 6
n_runs         = n_trajectories * (n_groups + 1)

assert morris_problem['num_vars'] == len(morris_problem['names']) == len(morris_problem['groups']) == len(morris_problem['bounds'])
assert n_runs == 66

# run numbering: continues the repo convention (R125-R128 used)
first_run = 129 
last_run = first_run + n_runs - 1
morris_seed          = 42      # SALib sampling reproducibility (NOT the NetLogo run-seed)
netlogo_run_seed    = 11      # NetLogo run-seed for ALL 66 runs (common random numbers)

# outputs
weights_out_dir      = ti_city_in_path          # run weights files land next to the template
scenarios_morris_path = ti_city_in_path / 'scenarios_calibration.csv'   # the Morris batch scenario table
manifest_path        = docs_dir / f"{case_city}_TA-City_model_R{first_run}_{last_run}_manifest.csv"
design_path          = docs_dir / f"{case_city}_TA-City_model_R{first_run}_{last_run}_design.csv"   # X matrix, exact reproduction
problem_path         = docs_dir / f"{case_city}_TA-City_model_R{first_run}_{last_run}_problem.json"

print(f'{morris_problem["num_vars"]} variables / {n_groups} groups / {n_trajectories} trajectories -> {n_runs} runs (R{first_run}-{last_run})')

25 variables / 10 groups / 6 trajectories -> 66 runs (R129-194)


# 2. Sample the design

`num_levels=4` gives Delta = p / (2(p-1)) = 2/3. The sampling seed is fixed so `X` is exactly reproducible; `morris_analyze.analyze` later needs this identical problem dict and X.

In [ ]:
X = morris_sample.sample(
    problem = morris_problem,
    N = n_trajectories,
    num_levels = 4,
    seed = morris_seed,
)

trajectory_id = np.repeat(np.arange(n_trajectories), n_groups + 1)  # 11 rows per trajectory
assert X.shape == (n_runs, morris_problem['num_vars'])
print('design matrix:', X.shape)

design_df = pd.DataFrame(X, columns=morris_problem['names'])
design_df.insert(0, 'trajectory', trajectory_id)
design_df.insert(1, 'run_num', first_run + np.arange(n_runs))
display(design_df.head(12))

# 3. Load templates

Weights: 6 agent rows x 17 columns, cloned from `weights_s0.csv` (BOM + trailing comment rows are stripped). Scenarios: schema v3 (14 columns); the s0 row is cloned and patched — household-number, names, exclusion_file and run-num conventions are inherited unchanged.

In [ ]:
weights_template = pd.read_csv(
    filepath_or_buffer = template_weights_path,
    encoding = 'utf-8-sig',
).dropna(subset = ['agent'])
weights_template = weights_template[weights_template['agent'].isin(['dp', 'li', 'mi', 'hi', 'mi_red', 'hi_red'])].reset_index(drop = True)
assert len(weights_template) == 6, f'expected 6 agent rows, got {len(weights_template)}'

scenarios_template = pd.read_csv(filepath_or_buffer = template_scenarios_path, encoding = 'utf-8-sig')
scenarios_template = scenarios_template.dropna(subset = [scenarios_template.columns[0]])
s0_row = scenarios_template[scenarios_template.iloc[:, 0] == template_scenario_name]
assert len(s0_row) == 1, f'expected exactly one {template_scenario_name} row in scenarios.csv'
print('weights template:', weights_template.shape, '| scenarios template:', scenarios_template.shape)

agent_rows   = ['dp', 'li', 'mi', 'hi', 'mi_red', 'hi_red']
weight_cols  = ['suburban', 'neighbour', 'bu', 'cbd', 'mall', 'markets', 'road', 'density',
                'schools', 'attractive', 'health', 'random', 'water', 'exposed']
hi_floor      = 5   # realized shares are normalized to sum 100 - hi_floor; hi = residual

# 4. Realization rules (writer rules 3, 4, 6, 7 of the spec; rule 5 revoked)

- **Shares** (rule 3): raw sampled values are NORMALIZED to sum `100 - hi_floor`, `hi = 100 - sum`. No clipping — clipping after sampling would invalidate the elementary effects (Y responds to the normalized quantity, which is monotone in the raw one).
- **rad** (rule 4): rounded to int, clipped to 1-5.
- **Direct writes everywhere else** (rule 6, strengthened after the RED bug): every sampled value is written into its cells exactly as sampled; no hidden nonlinear transforms. A transform is allowed only when feasibility forces it (share normalization) or on an integer grid (rad). The RED weight rows follow the same shared sampled weights as the household rows; `exposed` stays signed in [-5, 5] for all six rows.
- **nom-slope** (rule 7) written to the RED rows only (only RED utilities carry it in v41).

In [ ]:
def bake_weights(template_df=None, values=None):
    """Clone the s0 weights table and write the sampled values into their sites."""
    w = template_df.copy()

    # group 1-6: shared weight columns, all six agent rows
    for col, var in [('road', 'w_road'),
                     ('bu', 'w_bu'), ('density', 'w_density'),
                     ('suburban', 'w_suburban'), ('cbd', 'w_cbd'), ('attractive', 'w_attractive'),
                     ('mall', 'w_mall'), ('markets', 'w_markets'), ('schools', 'w_schools'), ('health', 'w_health'),
                     ('random', 'w_random'), ('water', 'w_water'),
                     ('neighbour', 'w_neighbour')]:
        w[col] = values[var]

    # group 6 exposure: SIGNED
    w['exposed'] = values['w_exposed']

    # group 7 terrain: nom-slope only on the RED rows
    w.loc[w['agent'] == 'mi_red', 'nom-slope'] = values['w_nom_slope']
    w.loc[w['agent'] == 'hi_red', 'nom-slope'] = values['w_nom_slope']

    # group 10 RED: the rad columns are the only RED-only utility knobs; the 14 weights
    # come from the same shared columns as the household rows above (direct writes)
    w.loc[w['agent'] == 'mi_red', 'rad'] = int(np.clip(round(values['red_rad_mi']), 1, 5))
    w.loc[w['agent'] == 'hi_red', 'rad'] = int(np.clip(round(values['red_rad_hi']), 1, 5))

    # invariant: exposure stays inside the signed instrument on ALL rows
    assert w['exposed'].between(-5, 5).all(), 'exposed outside [-5, 5]'
    assert w.loc[w['agent'].isin(agent_rows), weight_cols].apply(
        lambda col: col.between(-5, 10).all()).all(), 'weight outside instrument'

    return w


def bake_scenario_row(s0_series=None, values=None):
    """Clone the s0 scenarios.csv row and patch the sampled scalar + share columns (schema v3 indexes)."""
    row = s0_series.copy()

    # group 7 terrain
    row.iloc[6] = values['slope_coefficient']
    row.iloc[7] = values['critical_slope']

    # group 8 class shares, normalized (rule 3)
    raw = np.array([values['share_dp'], values['share_li'], values['share_mi']], dtype = float)
    realized = raw / raw.sum() * (100.0 - hi_floor)
    row.iloc[2], row.iloc[3], row.iloc[4] = realized
    row.iloc[5] = 100.0 - realized.sum()            # hi = residual

    # group 9 planning
    row.iloc[8] = values['development_control']

    # group 10 RED scalars
    # estate_developer_number is an agent COUNT -> INTEGER grid (same rule as rad):
    # a non-integer value reaches make-developers, where create-developers truncates it but the
    # n-of class counts are rounded from the RAW float -> off-by-one 'requested N random agents
    # from a set of only N-1' crashes (seen 2026-11-26 with estate=266.6667, pct_high=60: n-of 160 from 159)
    row.iloc[9]  = int(round(values['estate_developer_number']))
    row.iloc[10] = values['pct_high_end_developers']

    return row

# 5. Bake all 66 runs

One weights file + one scenarios row per run; run_num 129-194; scenario names `s_m001`...`s_m066`.

In [ ]:
records = []
scenarios_rows = []
s0_series = s0_row.iloc[0]

for i in range(n_runs):
    values  = design_df.iloc[i].to_dict()
    run_num = int(values['run_num'])
    scenario_name = f's_m{i + 1:03d}'
    weights_file = f'weights_s_m{i + 1:03d}.csv'

    w = bake_weights(template_df = weights_template, values = values)
    w.to_csv(weights_out_dir / weights_file, index = False, encoding = 'utf-8')

    row = bake_scenario_row(s0_series = s0_series, values = values)
    row.iloc[0]  = scenario_name
    row.iloc[11] = run_num                       # run_num column (schema v3, index 11, bare number)
    row.iloc[13] = f'data/in/{weights_file}'    # weights_file pointer: prefixed, v42 resolves relative to model/JAK/
    scenarios_rows.append(row)
    trajectory = int(values['trajectory'])
    point = i % (n_groups + 1)

    records.append({
        'offset': i,
        'scenario': scenario_name,
        'purpose': f'Morris pass-1 screening — trajectory {trajectory + 1}, point {point + 1}/11',
        'variation': f'parameter set R{run_num} in JAK_TA-City_model_R129-194_design.csv; weights baked in {weights_file}',
    })

scenarios_morris = pd.DataFrame(scenarios_rows, columns = scenarios_template.columns)
scenarios_morris.to_csv(scenarios_morris_path, index = False, encoding = 'utf-8-sig')

# internal view used by the QC + BehaviorSpace cells below (the OFFICIAL manifest is
# the ti_city_15-format CSV written by create_folders in section 5b)
batch_df = pd.DataFrame({
    'scenario': [r['scenario'] for r in records],
    'run_num': first_run + np.arange(n_runs),
    'weights_file': [f"weights_s_m{i + 1:03d}.csv" for i in range(n_runs)],
    'trajectory': design_df['trajectory'].values,
    'run_seed': netlogo_run_seed,
})
print(f'baked {n_runs} weights files + {scenarios_morris_path.name}; batch list ready for create_folders')
display(batch_df.head())

# 6. Create folders
`create_folders()` comes from `ti_city_00_helper_functions` (the same function that built the
R125-128 batch): it plans the folder tree, copies the .prj / scenarios / weights files into each
`run/seed/in` folder, and writes `docs/JAK_TA-model_manifest_R129_194.csv` with the full schema
(run, scenario, run_seed, purpose, parameter_variation, output_folder, code_version, inputs,
mode_switches, household_number, scenarios_csv_md5, weights_file_md5, date_created, status).
Inspect with the dry run first, then create.

In [ ]:
# ----------------------------------------------------------------------------
# Batch definition — R129+ Morris pass-1 group screening (v42 model,
# design v3.1: 10 groups / 25 variables / 6 trajectories / 66 runs,
# run-seed constant 11 across the whole design, common random numbers)
# ----------------------------------------------------------------------------
seeds_list = [11]
first_run = 129

code_version = 'v42 (post-R125-128 verification)'
inputs = "same corrected input vintage as R125-128 (all-roads layer + bu-proximity year 2000)"
model_mode = "calibrate? = TRUE, RED-on?, neighbour-on? on; export-per-tick? off"
hh_number = "frozen (cloned from the s0 scenario row, per scenarios.csv)"
status = "pending"

morris_runs_list = [
    {
        'offset': 0,
        'scenario': 's_m001',
        'purpose': 'Morris pass-1 screening — trajectory 1, point 1/11',
        'variation': 'parameter set R129 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m001.csv',
    },
    {
        'offset': 1,
        'scenario': 's_m002',
        'purpose': 'Morris pass-1 screening — trajectory 1, point 2/11',
        'variation': 'parameter set R130 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m002.csv',
    },
    {
        'offset': 2,
        'scenario': 's_m003',
        'purpose': 'Morris pass-1 screening — trajectory 1, point 3/11',
        'variation': 'parameter set R131 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m003.csv',
    },
    {
        'offset': 3,
        'scenario': 's_m004',
        'purpose': 'Morris pass-1 screening — trajectory 1, point 4/11',
        'variation': 'parameter set R132 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m004.csv',
    },
    {
        'offset': 4,
        'scenario': 's_m005',
        'purpose': 'Morris pass-1 screening — trajectory 1, point 5/11',
        'variation': 'parameter set R133 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m005.csv',
    },
    {
        'offset': 5,
        'scenario': 's_m006',
        'purpose': 'Morris pass-1 screening — trajectory 1, point 6/11',
        'variation': 'parameter set R134 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m006.csv',
    },
    {
        'offset': 6,
        'scenario': 's_m007',
        'purpose': 'Morris pass-1 screening — trajectory 1, point 7/11',
        'variation': 'parameter set R135 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m007.csv',
    },
    {
        'offset': 7,
        'scenario': 's_m008',
        'purpose': 'Morris pass-1 screening — trajectory 1, point 8/11',
        'variation': 'parameter set R136 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m008.csv',
    },
    {
        'offset': 8,
        'scenario': 's_m009',
        'purpose': 'Morris pass-1 screening — trajectory 1, point 9/11',
        'variation': 'parameter set R137 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m009.csv',
    },
    {
        'offset': 9,
        'scenario': 's_m010',
        'purpose': 'Morris pass-1 screening — trajectory 1, point 10/11',
        'variation': 'parameter set R138 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m010.csv',
    },
    {
        'offset': 10,
        'scenario': 's_m011',
        'purpose': 'Morris pass-1 screening — trajectory 1, point 11/11',
        'variation': 'parameter set R139 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m011.csv',
    },
    {
        'offset': 11,
        'scenario': 's_m012',
        'purpose': 'Morris pass-1 screening — trajectory 2, point 1/11',
        'variation': 'parameter set R140 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m012.csv',
    },
    {
        'offset': 12,
        'scenario': 's_m013',
        'purpose': 'Morris pass-1 screening — trajectory 2, point 2/11',
        'variation': 'parameter set R141 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m013.csv',
    },
    {
        'offset': 13,
        'scenario': 's_m014',
        'purpose': 'Morris pass-1 screening — trajectory 2, point 3/11',
        'variation': 'parameter set R142 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m014.csv',
    },
    {
        'offset': 14,
        'scenario': 's_m015',
        'purpose': 'Morris pass-1 screening — trajectory 2, point 4/11',
        'variation': 'parameter set R143 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m015.csv',
    },
    {
        'offset': 15,
        'scenario': 's_m016',
        'purpose': 'Morris pass-1 screening — trajectory 2, point 5/11',
        'variation': 'parameter set R144 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m016.csv',
    },
    {
        'offset': 16,
        'scenario': 's_m017',
        'purpose': 'Morris pass-1 screening — trajectory 2, point 6/11',
        'variation': 'parameter set R145 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m017.csv',
    },
    {
        'offset': 17,
        'scenario': 's_m018',
        'purpose': 'Morris pass-1 screening — trajectory 2, point 7/11',
        'variation': 'parameter set R146 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m018.csv',
    },
    {
        'offset': 18,
        'scenario': 's_m019',
        'purpose': 'Morris pass-1 screening — trajectory 2, point 8/11',
        'variation': 'parameter set R147 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m019.csv',
    },
    {
        'offset': 19,
        'scenario': 's_m020',
        'purpose': 'Morris pass-1 screening — trajectory 2, point 9/11',
        'variation': 'parameter set R148 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m020.csv',
    },
    {
        'offset': 20,
        'scenario': 's_m021',
        'purpose': 'Morris pass-1 screening — trajectory 2, point 10/11',
        'variation': 'parameter set R149 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m021.csv',
    },
    {
        'offset': 21,
        'scenario': 's_m022',
        'purpose': 'Morris pass-1 screening — trajectory 2, point 11/11',
        'variation': 'parameter set R150 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m022.csv',
    },
    {
        'offset': 22,
        'scenario': 's_m023',
        'purpose': 'Morris pass-1 screening — trajectory 3, point 1/11',
        'variation': 'parameter set R151 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m023.csv',
    },
    {
        'offset': 23,
        'scenario': 's_m024',
        'purpose': 'Morris pass-1 screening — trajectory 3, point 2/11',
        'variation': 'parameter set R152 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m024.csv',
    },
    {
        'offset': 24,
        'scenario': 's_m025',
        'purpose': 'Morris pass-1 screening — trajectory 3, point 3/11',
        'variation': 'parameter set R153 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m025.csv',
    },
    {
        'offset': 25,
        'scenario': 's_m026',
        'purpose': 'Morris pass-1 screening — trajectory 3, point 4/11',
        'variation': 'parameter set R154 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m026.csv',
    },
    {
        'offset': 26,
        'scenario': 's_m027',
        'purpose': 'Morris pass-1 screening — trajectory 3, point 5/11',
        'variation': 'parameter set R155 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m027.csv',
    },
    {
        'offset': 27,
        'scenario': 's_m028',
        'purpose': 'Morris pass-1 screening — trajectory 3, point 6/11',
        'variation': 'parameter set R156 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m028.csv',
    },
    {
        'offset': 28,
        'scenario': 's_m029',
        'purpose': 'Morris pass-1 screening — trajectory 3, point 7/11',
        'variation': 'parameter set R157 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m029.csv',
    },
    {
        'offset': 29,
        'scenario': 's_m030',
        'purpose': 'Morris pass-1 screening — trajectory 3, point 8/11',
        'variation': 'parameter set R158 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m030.csv',
    },
    {
        'offset': 30,
        'scenario': 's_m031',
        'purpose': 'Morris pass-1 screening — trajectory 3, point 9/11',
        'variation': 'parameter set R159 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m031.csv',
    },
    {
        'offset': 31,
        'scenario': 's_m032',
        'purpose': 'Morris pass-1 screening — trajectory 3, point 10/11',
        'variation': 'parameter set R160 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m032.csv',
    },
    {
        'offset': 32,
        'scenario': 's_m033',
        'purpose': 'Morris pass-1 screening — trajectory 3, point 11/11',
        'variation': 'parameter set R161 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m033.csv',
    },
    {
        'offset': 33,
        'scenario': 's_m034',
        'purpose': 'Morris pass-1 screening — trajectory 4, point 1/11',
        'variation': 'parameter set R162 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m034.csv',
    },
    {
        'offset': 34,
        'scenario': 's_m035',
        'purpose': 'Morris pass-1 screening — trajectory 4, point 2/11',
        'variation': 'parameter set R163 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m035.csv',
    },
    {
        'offset': 35,
        'scenario': 's_m036',
        'purpose': 'Morris pass-1 screening — trajectory 4, point 3/11',
        'variation': 'parameter set R164 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m036.csv',
    },
    {
        'offset': 36,
        'scenario': 's_m037',
        'purpose': 'Morris pass-1 screening — trajectory 4, point 4/11',
        'variation': 'parameter set R165 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m037.csv',
    },
    {
        'offset': 37,
        'scenario': 's_m038',
        'purpose': 'Morris pass-1 screening — trajectory 4, point 5/11',
        'variation': 'parameter set R166 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m038.csv',
    },
    {
        'offset': 38,
        'scenario': 's_m039',
        'purpose': 'Morris pass-1 screening — trajectory 4, point 6/11',
        'variation': 'parameter set R167 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m039.csv',
    },
    {
        'offset': 39,
        'scenario': 's_m040',
        'purpose': 'Morris pass-1 screening — trajectory 4, point 7/11',
        'variation': 'parameter set R168 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m040.csv',
    },
    {
        'offset': 40,
        'scenario': 's_m041',
        'purpose': 'Morris pass-1 screening — trajectory 4, point 8/11',
        'variation': 'parameter set R169 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m041.csv',
    },
    {
        'offset': 41,
        'scenario': 's_m042',
        'purpose': 'Morris pass-1 screening — trajectory 4, point 9/11',
        'variation': 'parameter set R170 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m042.csv',
    },
    {
        'offset': 42,
        'scenario': 's_m043',
        'purpose': 'Morris pass-1 screening — trajectory 4, point 10/11',
        'variation': 'parameter set R171 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m043.csv',
    },
    {
        'offset': 43,
        'scenario': 's_m044',
        'purpose': 'Morris pass-1 screening — trajectory 4, point 11/11',
        'variation': 'parameter set R172 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m044.csv',
    },
    {
        'offset': 44,
        'scenario': 's_m045',
        'purpose': 'Morris pass-1 screening — trajectory 5, point 1/11',
        'variation': 'parameter set R173 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m045.csv',
    },
    {
        'offset': 45,
        'scenario': 's_m046',
        'purpose': 'Morris pass-1 screening — trajectory 5, point 2/11',
        'variation': 'parameter set R174 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m046.csv',
    },
    {
        'offset': 46,
        'scenario': 's_m047',
        'purpose': 'Morris pass-1 screening — trajectory 5, point 3/11',
        'variation': 'parameter set R175 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m047.csv',
    },
    {
        'offset': 47,
        'scenario': 's_m048',
        'purpose': 'Morris pass-1 screening — trajectory 5, point 4/11',
        'variation': 'parameter set R176 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m048.csv',
    },
    {
        'offset': 48,
        'scenario': 's_m049',
        'purpose': 'Morris pass-1 screening — trajectory 5, point 5/11',
        'variation': 'parameter set R177 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m049.csv',
    },
    {
        'offset': 49,
        'scenario': 's_m050',
        'purpose': 'Morris pass-1 screening — trajectory 5, point 6/11',
        'variation': 'parameter set R178 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m050.csv',
    },
    {
        'offset': 50,
        'scenario': 's_m051',
        'purpose': 'Morris pass-1 screening — trajectory 5, point 7/11',
        'variation': 'parameter set R179 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m051.csv',
    },
    {
        'offset': 51,
        'scenario': 's_m052',
        'purpose': 'Morris pass-1 screening — trajectory 5, point 8/11',
        'variation': 'parameter set R180 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m052.csv',
    },
    {
        'offset': 52,
        'scenario': 's_m053',
        'purpose': 'Morris pass-1 screening — trajectory 5, point 9/11',
        'variation': 'parameter set R181 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m053.csv',
    },
    {
        'offset': 53,
        'scenario': 's_m054',
        'purpose': 'Morris pass-1 screening — trajectory 5, point 10/11',
        'variation': 'parameter set R182 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m054.csv',
    },
    {
        'offset': 54,
        'scenario': 's_m055',
        'purpose': 'Morris pass-1 screening — trajectory 5, point 11/11',
        'variation': 'parameter set R183 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m055.csv',
    },
    {
        'offset': 55,
        'scenario': 's_m056',
        'purpose': 'Morris pass-1 screening — trajectory 6, point 1/11',
        'variation': 'parameter set R184 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m056.csv',
    },
    {
        'offset': 56,
        'scenario': 's_m057',
        'purpose': 'Morris pass-1 screening — trajectory 6, point 2/11',
        'variation': 'parameter set R185 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m057.csv',
    },
    {
        'offset': 57,
        'scenario': 's_m058',
        'purpose': 'Morris pass-1 screening — trajectory 6, point 3/11',
        'variation': 'parameter set R186 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m058.csv',
    },
    {
        'offset': 58,
        'scenario': 's_m059',
        'purpose': 'Morris pass-1 screening — trajectory 6, point 4/11',
        'variation': 'parameter set R187 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m059.csv',
    },
    {
        'offset': 59,
        'scenario': 's_m060',
        'purpose': 'Morris pass-1 screening — trajectory 6, point 5/11',
        'variation': 'parameter set R188 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m060.csv',
    },
    {
        'offset': 60,
        'scenario': 's_m061',
        'purpose': 'Morris pass-1 screening — trajectory 6, point 6/11',
        'variation': 'parameter set R189 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m061.csv',
    },
    {
        'offset': 61,
        'scenario': 's_m062',
        'purpose': 'Morris pass-1 screening — trajectory 6, point 7/11',
        'variation': 'parameter set R190 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m062.csv',
    },
    {
        'offset': 62,
        'scenario': 's_m063',
        'purpose': 'Morris pass-1 screening — trajectory 6, point 8/11',
        'variation': 'parameter set R191 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m063.csv',
    },
    {
        'offset': 63,
        'scenario': 's_m064',
        'purpose': 'Morris pass-1 screening — trajectory 6, point 9/11',
        'variation': 'parameter set R192 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m064.csv',
    },
    {
        'offset': 64,
        'scenario': 's_m065',
        'purpose': 'Morris pass-1 screening — trajectory 6, point 10/11',
        'variation': 'parameter set R193 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m065.csv',
    },
    {
        'offset': 65,
        'scenario': 's_m066',
        'purpose': 'Morris pass-1 screening — trajectory 6, point 11/11',
        'variation': 'parameter set R194 in JAK_TA-City_model_R129-194_design.csv; weights baked in weights_s_m066.csv',
    },
]

In [ ]:
from ti_city_00_helper_functions import create_folders

create_folders(
    case_city = case_city,
    seeds_list = seeds_list,
    runs_list = morris_runs_list,
    first_run = first_run,
    run_type="dry-run",
    root_folder_path = git_path,
    inputs=inputs,
    code_version=code_version,
    model_mode=model_mode,
    hh_number=hh_number,
    status=status,
    scenarios_file = 'scenarios_calibration.csv'
)

In [ ]:
create_folders(
    case_city = case_city,
    seeds_list = seeds_list,
    runs_list = morris_runs_list,
    first_run = first_run,
    run_type="create",
    root_folder_path = git_path,
    inputs=inputs,
    code_version=code_version,
    model_mode=model_mode,
    hh_number=hh_number,
    status=status,
    scenarios_file = 'scenarios_calibration.csv'
)

# 6. Export the exact design (reproducibility)

`design.csv` (the X matrix + run_num/trajectory) and `problem.json` are what `morris_analyze.analyze(problem, X, Y)` needs later — keep them untouched.

In [ ]:
design_df.to_csv(design_path, index = False, encoding = 'utf-8-sig')

# cross-check: the folder/manifest side (create_folders) and the design side agree

assert manifest_path.exists(), f'create_folders manifest missing: {manifest_path}'

problem_json = {**morris_problem, 'n_trajectories': n_trajectories, 'num_levels': 4,
                'sampling_seed': morris_seed, 'netlogo_run_seed': netlogo_run_seed,
                'run_num_range': [first_run, last_run]}
Path(problem_path).write_text(json.dumps(problem_json, indent = 2), encoding = 'utf-8')
print(f'exported: {design_path.name} | {problem_path.name} | {manifest_path.name} | {scenarios_morris_path.name}')

# 7. QC the batch

Reloads written files and checks the writer rules + the Phase 0 invariants (household-number cloning, share sums, integer rad, signed exposure bounds, weights grid integrity).

In [ ]:
# re-read what was written
check_scen = pd.read_csv(filepath_or_buffer = scenarios_morris_path, encoding = 'utf-8-sig')
assert len(check_scen) == n_runs

# share sums / hi residual
share_sum = check_scen.iloc[:, [2, 3, 4, 5]].sum(axis = 1)
assert (share_sum.round(6) == 100).all(), check_scen[(share_sum.round(6) != 100)]
assert (check_scen.iloc[:, 5] >= hi_floor - 1e-9).all(), 'hi below floor'

# household-number cloned from s0, scalar ranges
assert (check_scen.iloc[:, 1].astype(str) == s0_series.iloc[1]).all(), 'household-number drifted'
assert check_scen.iloc[:, 6].between(20, 100).all() and check_scen.iloc[:, 7].between(5, 30).all()
assert check_scen.iloc[:, 8].between(30, 90).all()
assert check_scen.iloc[:, 9].between(100, 600).all() and check_scen.iloc[:, 10].between(10, 60).all()
assert (check_scen.iloc[:, 9] % 1 == 0).all(), 'estate-developer-number not an integer (agent-count grid)'

# run manifest (ti_city_15 format) matches the design batch_df
check_manifest = pd.read_csv(filepath_or_buffer = morris_manifest_path, encoding = 'utf-8-sig')
assert len(check_manifest) == n_runs, 'manifest run count mismatch'
assert (check_manifest['run_seed'] == netlogo_run_seed).all(), 'manifest seed drift'
assert (check_manifest['scenario'].astype(str) == batch_df['scenario'].astype(str)).all(), 'manifest scenario drift'
for _, r in batch_df.iterrows():          # every manifest entry points at an existing output folder
    folder = Path(git_path) / f"model/{case_city}/data/out/R{int(r['run_num'])}/seed{netlogo_run_seed}"
    assert folder.exists(), f'missing folder {folder}'

# weights files
failures = 0
for _, r in batch_df.iterrows():
    w = pd.read_csv(filepath_or_buffer = weights_out_dir / r['weights_file'],
                    encoding = 'utf-8-sig').dropna(subset = ['agent'])
    ok = (len(w) == 6
          and set(w['agent']) == {'dp', 'li', 'mi', 'hi', 'mi_red', 'hi_red'}
          and (w['maxof'] == 2).all()
          and w['rad'].between(1, 5).all() and (w['rad'] % 1 == 0).all()
          and w['exposed'].between(-5, 5).all()
          and w.loc[w['agent'].isin(['mi_red', 'hi_red']), 'nom-slope'].between(0, 10).all())
    failures += (not ok)
assert failures == 0, f'{failures} weights files failed QC'

# how many cells actually move vs the template (sanity: not all runs identical)
first = pd.read_csv(filepath_or_buffer = weights_out_dir / 'weights_s_m001.csv',
                    encoding = 'utf-8-sig').dropna(subset = ['agent'])
changed = (first[weight_cols].values != weights_template[weight_cols].values).sum()
print(f'QC PASS: {n_runs} runs | share sums 100 | household-number frozen | rad int 1-5 | '
      f's_m001 vs s0 template: {changed}/{len(weight_cols) * 6} weight cells changed')
display(check_scen.head())

# 9. BehaviorSpace experiment for the batch

Single experiment, `scenario` varied in the 66-run list, `run-seed` constant 11, repetitions 1. Paste into the model file's experiments section (or run headless via the generated XML in docs/).

In [ ]:
morris_metric_list = [
    'behaviorspace-run-number',
    'scenario',
    'run-seed',
    '(count patches with [new-urban = 1])',
    '(household-number - count patches with [new-urban = 1])',
    '(sum [estate-size] of developers with [class = 2 and MED-movement = 1]) - count patches with [pMED-movement = 1]',
    '(sum [estate-size] of developers with [class = 3 and HED-movement = 1]) - count patches with [pHED-movement = 1]',
    'count patches with [pMED-movement = 1]',
    'count patches with [pHED-movement = 1]',
    'sum [estate-size] of developers with [(class = 2 and MED-movement = 0) or (class = 3 and HED-movement = 0)]',
    'ticks',
]
metrics_xml = '\n'.join(
    f'      <metric>{m}</metric>' for m in morris_metric_list
)
scenario_values_xml = '\n'.join(
    f'        <value value="&quot;{s}&quot;"></value>' for s in batch_df['scenario']
)

# netLogo 7.0.3 Lab (BehaviorSpace) XML document format, mirroring the
# NetLogo-exported JAK_TA-model_BS_experiment_R125-128.xml one-for-one:
# <experiments> root; metrics/constants wrappers; runMetricsEveryStep + timeLimit
# attributes; run-seed set listed before scenario; full metric list incl. the
# RED residual/QC reporters (mandatory here: estate-developer-number, pct-high-end
# and rad are Morris variables, so the residual invariants must be watchable).
experiment_xml = f'''<experiments>
  <experiment name="TA_city_morris_R129-194" repetitions="1" sequentialRunOrder="true" runMetricsEveryStep="true" timeLimit="100">
    <setup>set-up</setup>
    <go>go</go>
    <exitCondition><![CDATA[ticks >= simulation-period]]></exitCondition>
    <metrics>
{metrics_xml}
    </metrics>
    <constants>
      <enumeratedValueSet variable="run-seed">
        <value value="{netlogo_run_seed}"></value>
      </enumeratedValueSet>
      <enumeratedValueSet variable="scenario">
{scenario_values_xml}
      </enumeratedValueSet>
      <enumeratedValueSet variable="calibrate?">
        <value value="true"></value>
      </enumeratedValueSet>
      <enumeratedValueSet variable="RED-on?">
        <value value="true"></value>
      </enumeratedValueSet>
      <enumeratedValueSet variable="neighbour-on?">
        <value value="true"></value>
      </enumeratedValueSet>
      <enumeratedValueSet variable="track-neighbourhood-changes?">
        <value value="false"></value>
      </enumeratedValueSet>
      <enumeratedValueSet variable="export-per-tick?">
        <value value="false"></value>
      </enumeratedValueSet>
    </constants>
  </experiment>
</experiments>'''

experiment_path = docs_dir / 'JAK_TA-City_model_R129-194_bs-experiment.xml'
Path(experiment_path).write_text(experiment_xml, encoding = 'utf-8')

assert experiment_xml.count('value="&quot;s_m') == n_runs
assert '<metrics>' in experiment_xml and '<constants>' in experiment_xml and experiment_xml.startswith('<experiments>')
print(f'wrote {experiment_path.name}: {n_runs} scenario values, run-seed {netlogo_run_seed} constant, '
      f'{len(morris_metric_list)} metrics')

# The end (for now).

After the batch: `morris_analyze.analyze(problem = morris_problem, X = X, Y = Y)` once per Y column (strict FoM, FuzzyFoM t1/t2/t5/t3, HitRate, QuantityRatio, informality share), Y keyed by run_num. Rank groups by mu*, read sigma for interactions; confirm top groups with run-seeds 22/33.